# 结构化输出
## Pydantic

In [ ]:
from anthropic import BaseModel
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

from pydantic import Field

# 从.env文件中加载环境变量
load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_API_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    model="deepseek-chat",
    # api_key = DEEPSEEK_API_KEY,
    # url = DEEPSEEK_API_URL
    extra_body={
        "thinking": {
            "type": "disabled"  # 🔑 关键：关闭思考模式
        }
    }
)


# 定义 Pydantic 模型
class Person(BaseModel):
    """人物休息"""
    name: str = Field(description="姓名")
    # 默认 为 1岁
    # age: int = Field(1,description="年龄")
    age: int = Field(description="年龄")
    occupation: str = Field(description="职业")


#使用 with_structured_output 即可引导模型进行结构化输出
structured_llm = model.with_structured_output(Person)
result = structured_llm.invoke("张三是一名乞丐,今年54岁")



In [ ]:
print(result)
print(result.name)
print(result.age)
print(result.occupation)

In [ ]:
from pydantic import BaseModel, Field, SecretStr


class MovieModel(BaseModel):
    """
    电影的详细信息
    """
    title: str = Field(description="电影标题")
    year: int = Field(description="电影上映年份")
    director: str = Field(description="导演")
    actor: str = Field(description="主要演员")
    rating: float = Field(description="电影评分，满分十分")


model_with_structure = model.with_structured_output(MovieModel)
response = model_with_structure.invoke(
    "请提供《我不是药神》的完整信息，包括：标题、上映年份、导演、主要演员饰演的角色、评分（满分10分）")

print(response)


In [ ]:
class Basketball(BaseModel):
    """球员分析"""
    player: str = Field(description="球员全名称")
    age: int = Field(description="年龄")
    country: str = Field("球员的国籍")
    tall: float = Field(description="身高")
    lifing: str = Field(description="生涯")


model_with_structure = model.with_structured_output(Basketball)
res = model_with_structure.invoke("请提供湖人的24号选手，包括年龄，国籍，身高，职业生涯")

print(res)

In [ ]:
from enum import Enum
from typing import Optional
from pydantic import BaseModel, Field
from rich import print as rprint


# 定义你的优先级枚举类
class Priority(str, Enum):
    LOW = "低"
    MEDIUM = "中"
    HIGH = "高"


class CustomerInfo(BaseModel):
    """客户信息"""
    name: str = Field(description="客户姓名")
    phone: str = Field(description="电话号码")
    email: Optional[str] = Field(description="邮箱")
    issue: str = Field(description="问题描述")
    urgency: Priority = Field(description="紧急程度")


structured_llm = model.with_structured_output(CustomerInfo)
conversation = """
客服: 您好，请问有什么可以帮助您？
客户: 我是王小明，电话 138-1234-5678，我的订单预计2个月后到达是吗？
客服: 好的，我帮您查一下
"""
res = model.invoke(f"从以下客服对话中提取客户信息以及紧急程度：\n{conversation}")
print("\n提取结果：")

rprint(res)


In [ ]:
from typing import List


class Person(BaseModel):
    """人物信息"""
    name: str
    age: int


class PersonList(BaseModel):
    """人物列表信息"""
    people: List[Person]


structured_llm = model.with_structured_output(PersonList)
result = structured_llm.invoke("张三 30岁，李四 25岁")
rprint(result)


In [ ]:
class Review(BaseModel):
    """产品评论"""
    product: str
    rating: int = Field(description="评分 1-5")
    pros: List[str] = Field(description="优点列表")
    cons: List[str] = Field(description="缺点列表")
structured_llm = model.with_structured_output(Review)
review = structured_llm.invoke("""iPhone 17 很棒！摄像头强大，手感好。但是价格贵，没有充电器。4分。
""")
    

rprint(review)
